# VVP Validation & Verification Protocol Tutorial

This tutorial demonstrates SIM-ONE's VVP protocol for validating logical rule structures before they're used in reasoning operations.

## What You'll Learn
- Validate logical rule syntax and structure
- Check for rule conflicts and inconsistencies
- Verify rule completeness and coverage
- Ensure rules are properly formed for reasoning engines

## Why Validate Rules?

Before performing logical reasoning (especially deductive reasoning), it's critical to ensure that:

1. **Syntax is correct** - Rules follow proper logical format
2. **No conflicts exist** - Rules don't contradict each other
3. **Structure is complete** - All necessary components are present
4. **Semantics are valid** - Rules make logical sense

## Use Cases

- **Rule-based systems** - Validate expert system rules before deployment
- **Deductive reasoning** - Ensure reasoning rules are properly formed
- **Policy enforcement** - Verify compliance rules are consistent
- **Logic programming** - Check logical programs for errors

## Understanding VVP execute() Method

The VVP protocol validates rule structures to ensure they're properly formed for use in logical reasoning operations.

### Parameters Table

| Parameter | Type | Required | Default | Description |
|-----------|------|----------|---------|-------------|
| **rules** | dict, list | Yes | - | Rules data to validate. Can be dict with "rules" key or list of rules directly |
| **context** | dict | No | None | Optional context for domain-specific validation |

### Rules Parameter Format

Rules must follow this exact structure:
```python
[
    [premises_list],  # List of premise strings (at least 1)
    conclusion        # Conclusion string (non-empty)
]
```

**Valid Examples**:
```python
# Single premise rule
[["All humans are mortal"], "Socrates is mortal"]

# Multiple premises rule
[["All birds have feathers", "Penguins are birds"], "Penguins have feathers"]

# Conditional rule
[["If it rains, ground is wet", "It is raining"], "Ground is wet"]
```

**Invalid Examples**:
```python
[[], "Conclusion"]                    # Empty premises list
[["Premise"], ""]                     # Empty conclusion
["Premise", "Conclusion"]             # Wrong structure - not nested
[["Premise"]]                         # Missing conclusion
```

### Context Parameter (Optional)

| Field | Type | Description |
|-------|------|-------------|
| **domain** | string | Domain name for specialized validation |
| **expected_coverage** | list | Areas/cases that rules should cover |
| **allow_conflicts** | bool | Whether conflicting rules are permitted (default: False) |
| **strict_mode** | bool | Enable stricter validation checks (default: False) |

### Return Structure

VVP returns a comprehensive validation dictionary:

```python
{
    "status": str,              # "valid", "invalid", or "warning"
    "total_rules": int,         # Total number of rules
    "valid_rules": int,         # Number of valid rules
    "invalid_rules": int,       # Number of invalid rules
    "validation_details": [...],# Per-rule validation results
    "conflicts": [...],         # Detected conflicts between rules
    "errors": [...],            # List of critical errors
    "warnings": [...],          # List of non-critical warnings
    "suggestions": [...],       # Improvement suggestions
    "completeness": {           # Coverage analysis (if context provided)
        "expected_coverage": [...],
        "covered": [...],
        "missing": [...],
        "coverage_percentage": float
    }
}
```

### Best Practices

1. **Always validate before reasoning** - Run VVP before using rules in REP
2. **Provide context** - Include domain and expected_coverage for better validation
3. **Check all error types** - Review errors, warnings, and conflicts
4. **Use proper structure** - Ensure rules follow [[premises], conclusion] format
5. **Test incrementally** - Validate as you build rule sets
6. **Handle all statuses** - Have logic for "valid", "invalid", and "warning" results

In [ ]:
# Standard library imports
import sys
import json
from pathlib import Path
from typing import Dict, Any, Optional, List
import pandas as pd

# Add SIM-ONE to Python path
SIMONE_ROOT = Path("../code").resolve()
sys.path.insert(0, str(SIMONE_ROOT))

# Import VVP protocol directly
from mcp_server.protocols.vvp.vvp import VVP

print("[OK] VVP (Validation & Verification Protocol) initialized")

## VVP Validation Pipeline

VVP follows a systematic validation process to ensure rule quality:

```
Input Rules + Context
    (down)
Parse Rule Structure
    (down)
Validate Each Rule Independently
    |-> Structure Check (syntax)
    |-> Semantic Check (logical validity)
    |-> Format Check (proper nesting)
    (down)
Cross-Rule Analysis
    |-> Conflict Detection (contradictions)
    |-> Completeness Check (coverage)
    |-> Dependency Analysis (rule chains)
    (down)
Generate Validation Report
    (down)
Return Results
```

### Pipeline Stages Explained

**Stage 1: Parse Rule Structure**
- Extract rules from input data
- Identify premises and conclusions
- Prepare for validation

**Stage 2: Validate Each Rule Independently**
- **Structure Check**: Ensures `[[[premises], conclusion]]` format
- **Semantic Check**: Verifies logical components make sense
- **Format Check**: Confirms proper list nesting and non-empty strings

**Stage 3: Cross-Rule Analysis**
- **Conflict Detection**: Compares rules to find contradictions
- **Completeness Check**: Analyzes coverage against expected domains
- **Dependency Analysis**: Identifies rule chains and circular dependencies

**Stage 4: Generate Validation Report**
- Compiles all validation results
- Creates per-rule details
- Generates error and warning lists
- Produces suggestions for improvement

**Stage 5: Return Results**
- Returns comprehensive validation dictionary
- Includes status, counts, details, conflicts, errors, warnings, suggestions

# Define rule set for checking completeness
rules_with_gaps = {
    "rules": [
        [["Temperature > 0 deg C", "Pressure = 1 atm"], "Water is liquid"],
        [["Temperature > 100 deg C", "Pressure = 1 atm"], "Water is gas"],
        # Missing: What about temperature < 0 deg C? (water is solid/ice)
    ],
    "context": {
        "domain": "Water phase transitions",
        "expected_coverage": ["solid", "liquid", "gas"]
    }
}

# Validate rules
result = vvp.execute(rules_with_gaps)

print("VVP Completeness Check:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")

if result.get("completeness"):
    comp = result["completeness"]
    print(f"\nCompleteness Analysis:")
    print(f"  Coverage: {comp.get('coverage_percentage', 0):.1f}%")
    print(f"  Covered Cases: {comp.get('covered_cases', [])}")
    print(f"  Missing Cases: {comp.get('missing_cases', [])}")

if result.get("suggestions"):
    print(f"\n[TIP] Suggestions for Improvement:")
    for suggestion in result["suggestions"]:
        print(f"  - {suggestion}")

## VVP as MCP Server Tool

VVP is available as an MCP (Model Context Protocol) server tool, making rule validation accessible to AI systems and automated workflows.

### Why VVP in MCP?

**Problem**: AI systems that work with logical rules need to ensure those rules are structurally valid and logically consistent before using them in reasoning operations. Invalid rules can cause:
- Reasoning failures
- Incorrect conclusions
- System errors
- Wasted computation on malformed logic

**Solution**: VVP validates rule structures before they're used, catching errors early and ensuring logical consistency.

### MCP Tool Integration

VVP is exposed through the SIM-ONE MCP server with two main tools:

#### 1. `vvp_validate_rule_structure`
Validates rule structures using the VVP protocol.

**Parameters**:
- `rules_data`: Path to JSON file with rules OR JSON string with rules
- `out_prefix`: Output file prefix (optional)
- `save_results`: Whether to save validation results to file (default: true)

**Use Case**: Standalone validation of rule sets before use.

**Example**:
```python
# Via MCP
result = mcp_client.call_tool(
    "vvp_validate_rule_structure",
    rules_data='{"rules": [[[\"All humans are mortal\", \"Socrates is a human\"], \"Socrates is mortal\"]]}',
    save_results=True
)
```

#### 2. `vvp_validate_rules_for_reasoning`
Validates rules with VVP and prepares them for REP deductive reasoning integration.

**Parameters**:
- `rules_data`: Path to JSON file or JSON string with rules
- `reasoning_context`: Context describing the reasoning domain (default: "General reasoning")
- `out_prefix`: Output file prefix (optional)
- `save_validated_rules`: Whether to save rules for REP integration (default: true)

**Use Case**: Validation + preparation for immediate use in REP reasoning.

**Example**:
```python
# Via MCP - validate and prepare for REP
result = mcp_client.call_tool(
    "vvp_validate_rules_for_reasoning",
    rules_data=rules_json,
    reasoning_context="Medical diagnosis",
    save_validated_rules=True
)

# If valid, rules are now ready for REP
if result["validation"]["status"] == "valid":
    rep_result = mcp_client.call_tool(
        "rep_perform_deductive_reasoning",
        facts=facts,
        rules=result["rep_ready_rules"]
    )
```

### VVP + REP Integration Pattern

The typical workflow for safe deductive reasoning:

```
1. Define Rules
   (down)
2. VVP Validation (vvp_validate_rules_for_reasoning)
   (down)
3. Check Status
   |-> valid -> Proceed to REP
   |-> warning -> Review conflicts, then proceed
   |-> invalid -> Fix errors, return to step 2
   (down)
4. REP Deductive Reasoning (rep_perform_deductive_reasoning)
   (down)
5. Get Conclusions
```

**Why This Matters**:
- **Safety**: Prevents malformed rules from causing reasoning failures
- **Quality**: Identifies logical conflicts before they produce contradictory conclusions
- **Efficiency**: Catches errors early instead of during reasoning
- **Clarity**: Provides clear feedback on what needs to be fixed

### Real-World Use Cases

#### 1. Expert Systems
Before deploying expert system rules for decision-making:
```python
# Validate medical diagnosis rules
vvp.execute(
    rules=medical_rules,
    context={"domain": "medical diagnosis"}
)
# Then use validated rules in expert system
```

#### 2. Policy Engines
Validate business policy rules for consistency:
```python
# Check approval workflow rules
vvp.execute(
    rules=approval_rules,
    context={"expected_coverage": ["request", "approval", "rejection"]}
)
# Ensures all policy cases are covered
```

#### 3. Automated Reasoning Pipelines
In AI workflows that generate and use rules:
```python
# AI generates rules from data
generated_rules = ai_system.extract_rules(data)

# VVP validates before use
validation = vvp.execute(rules=generated_rules)

if validation["status"] == "valid":
    # Use rules in downstream reasoning
    rep.perform_reasoning(rules=generated_rules, ...)
else:
    # Refine rule generation
    ai_system.refine_rules(validation["errors"])
```

#### 4. Logic Programming
Validate Prolog-style rule sets before execution:
```python
# Convert and validate logic programming rules
prolog_rules = convert_to_vvp_format(prolog_code)
validation = vvp.execute(rules=prolog_rules)
# Catch syntax and logic errors before runtime
```

### Integration Benefits

| Benefit | Description |
|---------|-------------|
| **Early Error Detection** | Catch rule errors before reasoning fails |
| **Logical Consistency** | Identify contradictions and conflicts |
| **Coverage Analysis** | Ensure rules cover expected cases |
| **Automated Workflows** | Enable AI systems to validate their own generated rules |
| **Quality Assurance** | Provide objective metrics for rule quality |

### When to Use VVP via MCP

Use VVP through the MCP server when:
- [OK] AI system generates rules that need validation
- [OK] Rules come from external sources (users, files, APIs)
- [OK] Building automated reasoning pipelines
- [OK] Rules will be used in REP deductive reasoning
- [OK] Need to ensure logical consistency before deployment
- [OK] Working with large rule sets that may have conflicts

In [ ]:
# Define valid rule structure
rules_data = {
    "rules": [
        [["All mammals are warm-blooded", "Dogs are mammals"], "Dogs are warm-blooded"],
        [["All birds have feathers", "Penguins are birds"], "Penguins have feathers"],
        [["If it rains, ground is wet", "It is raining"], "Ground is wet"]
    ]
}

# Validate rules using VVP
result = vvp.execute(rules_data)

print("VVP Validation Result:")
print("=" * 80)
print(json.dumps(result, indent=2))

print(f"\n[OK] Validation Status: {result.get('status', 'Unknown')}")
print(f"  Total Rules: {result.get('total_rules', 0)}")
print(f"  Valid Rules: {result.get('valid_rules', 0)}")

if result.get("validation_details"):
    print(f"\n[LIST] Validation Details:")
    for i, detail in enumerate(result["validation_details"], 1):
        print(f"  Rule {i}: {detail.get('status', 'Unknown')}")
        if detail.get("message"):
            print(f"    Message: {detail['message']}")

In [ ]:
# Define rules with various structural issues
invalid_rules = {
    "rules": [
        # Valid rule
        [["All cats are animals"], "Fluffy is an animal"],

        # INVALID: Empty premises list
        [[], "Something is true"],

        # INVALID: Missing conclusion (empty string)
        [["Some premise"], ""],

        # INVALID: Wrong structure (not nested list)
        ["premise", "conclusion"],

        # Valid rule
        [["X implies Y", "X is true"], "Y is true"]
    ]
}

# Validate rules
result = vvp.execute(invalid_rules)

print("VVP Validation Result (Invalid Rules):")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")
print(f"Valid Rules: {result.get('valid_rules', 0)}")
print(f"Invalid Rules: {result.get('invalid_rules', 0)}")

if result.get("validation_details"):
    print(f"\n[LIST] Detailed Validation:")
    for i, detail in enumerate(result["validation_details"], 1):
        status_symbol = "[OK]" if detail.get("status") == "valid" else "[X]"
        print(f"  {status_symbol} Rule {i}: {detail.get('status', 'Unknown')}")
        if detail.get("error"):
            print(f"    Error: {detail['error']}")
        if detail.get("message"):
            print(f"    Message: {detail['message']}")

if result.get("errors"):
    print(f"\n[X] Errors Summary:")
    for error in result["errors"]:
        print(f"  - {error}")

In [ ]:
# Define rules with various structural issues
invalid_rules = {
    "rules": [
        # Valid rule
        [["All cats are animals"], "Fluffy is an animal"],

        # INVALID: Empty premises list
        [[], "Something is true"],

        # INVALID: Missing conclusion (empty string)
        [["Some premise"], ""],

        # INVALID: Wrong structure (not nested list)
        ["premise", "conclusion"],

        # Valid rule
        [["X implies Y", "X is true"], "Y is true"]
    ]
}

# Validate rules
result = vvp.execute(invalid_rules)

print("VVP Validation Result (Invalid Rules):")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")
print(f"Valid Rules: {result.get('valid_rules', 0)}")
print(f"Invalid Rules: {result.get('invalid_rules', 0)}")

if result.get("validation_details"):
    print(f"\n[LIST] Detailed Validation:")
    for i, detail in enumerate(result["validation_details"], 1):
        status_symbol = "[OK]" if detail.get("status") == "valid" else "[X]"
        print(f"  {status_symbol} Rule {i}: {detail.get('status', 'Unknown')}")
        if detail.get("error"):
            print(f"    Error: {detail['error']}")
        if detail.get("message"):
            print(f"    Message: {detail['message']}")

if result.get("errors"):
    print(f"\n[X] Errors Summary:")
    for error in result["errors"]:
        print(f"  - {error}")

In [ ]:
# Define rules that conflict with each other
conflicting_rules = {
    "rules": [
        [["All birds can fly"], "Penguins can fly"],
        [["Penguins are birds"], "Penguins exist"],
        [["Penguins cannot fly"], "Penguins are flightless"],  # Conflicts with rule 1
        [["If X can fly, X has wings"], "Penguins have wings"]
    ]
}

# Validate rules
result = vvp.execute(conflicting_rules)

print("VVP Conflict Detection:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")

if result.get("conflicts"):
    print(f"\n[!] Conflicts Detected ({len(result['conflicts'])}):")
    for i, conflict in enumerate(result["conflicts"], 1):
        print(f"\n  Conflict {i}:")
        print(f"    Rule A: {conflict.get('rule_a')}")
        print(f"    Rule B: {conflict.get('rule_b')}")
        print(f"    Issue: {conflict.get('description')}")
else:
    print("\n[OK] No conflicts detected")

if result.get("warnings"):
    print(f"\n[TIP] Warnings:")
    for warning in result["warnings"]:
        print(f"  - {warning}")

# Define rules that conflict with each other
conflicting_rules = {
    "rules": [
        [["All birds can fly"], "Penguins can fly"],
        [["Penguins are birds"], "Penguins exist"],
        [["Penguins cannot fly"], "Penguins are flightless"],  # Conflicts with rule 1
        [["If X can fly, X has wings"], "Penguins have wings"]
    ]
}

# Validate rules
result = vvp.execute(conflicting_rules)

print("VVP Conflict Detection:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")

if result.get("conflicts"):
    print(f"\n[!] Conflicts Detected ({len(result['conflicts'])}):")
    for i, conflict in enumerate(result["conflicts"], 1):
        print(f"\n  Conflict {i}:")
        print(f"    Rule A: {conflict.get('rule_a')}")
        print(f"    Rule B: {conflict.get('rule_b')}")
        print(f"    Issue: {conflict.get('description')}")
else:
    print("\n[OK] No conflicts detected")

if result.get("warnings"):
    print(f"\n[TIP] Warnings:")
    for warning in result["warnings"]:
        print(f"  - {warning}")

In [ ]:
# Define rule set for checking completeness
rules_with_gaps = {
    "rules": [
        [["Temperature > 0 deg C", "Pressure = 1 atm"], "Water is liquid"],
        [["Temperature > 100 deg C", "Pressure = 1 atm"], "Water is gas"],
        # Missing: What about temperature < 0 deg C? (water is solid/ice)
    ],
    "context": {
        "domain": "Water phase transitions",
        "expected_coverage": ["solid", "liquid", "gas"]
    }
}

# Validate rules
result = vvp.execute(rules_with_gaps)

print("VVP Completeness Check:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")

if result.get("completeness"):
    comp = result["completeness"]
    print(f"\nCompleteness Analysis:")
    print(f"  Coverage: {comp.get('coverage_percentage', 0):.1f}%")
    print(f"  Covered Cases: {comp.get('covered_cases', [])}")
    print(f"  Missing Cases: {comp.get('missing_cases', [])}")

if result.get("suggestions"):
    print(f"\n[TIP] Suggestions for Improvement:")
    for suggestion in result["suggestions"]:
        print(f"  - {suggestion}")

# Define rule set for checking completeness
rules_with_gaps = {
    "rules": [
        [["Temperature > 0 deg C", "Pressure = 1 atm"], "Water is liquid"],
        [["Temperature > 100 deg C", "Pressure = 1 atm"], "Water is gas"],
        # Missing: What about temperature < 0 deg C? (water is solid/ice)
    ],
    "context": {
        "domain": "Water phase transitions",
        "expected_coverage": ["solid", "liquid", "gas"]
    }
}

# Validate rules
result = vvp.execute(rules_with_gaps)

print("VVP Completeness Check:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")

if result.get("completeness"):
    comp = result["completeness"]
    print(f"\nCompleteness Analysis:")
    print(f"  Coverage: {comp.get('coverage_percentage', 0):.1f}%")
    print(f"  Covered Cases: {comp.get('covered_cases', [])}")
    print(f"  Missing Cases: {comp.get('missing_cases', [])}")

if result.get("suggestions"):
    print(f"\n[TIP] Suggestions for Improvement:")
    for suggestion in result["suggestions"]:
        print(f"  - {suggestion}")

In [ ]:
# Create sample rules file
sample_rules_file = Path("sample_rules.json")

sample_rules = {
    "rules": [
        [["All employees must clock in", "John is an employee"], "John must clock in"],
        [["Meeting starts at 9 AM", "It is 9 AM"], "Meeting is starting"],
        [["If urgent, escalate immediately"], "Handle urgent matters promptly"]
    ]
}

# Write to file
sample_rules_file.write_text(json.dumps(sample_rules, indent=2))

# Validate rules from file
rules_from_file = json.loads(sample_rules_file.read_text())
result = vvp.execute(rules_from_file)

print(f"VVP Validation from File: {sample_rules_file}")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")
print(f"Valid Rules: {result.get('valid_rules', 0)}")

# Create validation summary table
if result.get("validation_details"):
    summary_data = []
    for i, detail in enumerate(result["validation_details"], 1):
        rule_text = f"Rule {i}"
        if detail.get("rule"):
            premises = detail["rule"].get("premises", [])
            conclusion = detail["rule"].get("conclusion", "")
            rule_text = f"{' & '.join(premises[:2])} -> {conclusion[:30]}"

        summary_data.append({
            "Rule": f"Rule {i}",
            "Status": detail.get("status", "Unknown"),
            "Structure": "[OK]" if detail.get("structure_valid") else "[X]",
            "Semantics": "[OK]" if detail.get("semantics_valid") else "[X]"
        })

    df = pd.DataFrame(summary_data)
    print("\n" + df.to_string(index=False))

# Cleanup
sample_rules_file.unlink()
print(f"\n[OK] Validation complete, file cleaned up")

In [ ]:
# Create sample rules file
sample_rules_file = Path("sample_rules.json")

sample_rules = {
    "rules": [
        [["All employees must clock in", "John is an employee"], "John must clock in"],
        [["Meeting starts at 9 AM", "It is 9 AM"], "Meeting is starting"],
        [["If urgent, escalate immediately"], "Handle urgent matters promptly"]
    ]
}

# Write to file
sample_rules_file.write_text(json.dumps(sample_rules, indent=2))

# Validate rules from file
rules_from_file = json.loads(sample_rules_file.read_text())
result = vvp.execute(rules_from_file)

print(f"VVP Validation from File: {sample_rules_file}")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")
print(f"Valid Rules: {result.get('valid_rules', 0)}")

# Create validation summary table
if result.get("validation_details"):
    summary_data = []
    for i, detail in enumerate(result["validation_details"], 1):
        rule_text = f"Rule {i}"
        if detail.get("rule"):
            premises = detail["rule"].get("premises", [])
            conclusion = detail["rule"].get("conclusion", "")
            rule_text = f"{' & '.join(premises[:2])} -> {conclusion[:30]}"

        summary_data.append({
            "Rule": f"Rule {i}",
            "Status": detail.get("status", "Unknown"),
            "Structure": "[OK]" if detail.get("structure_valid") else "[X]",
            "Semantics": "[OK]" if detail.get("semantics_valid") else "[X]"
        })

    df = pd.DataFrame(summary_data)
    print("\n" + df.to_string(index=False))

# Cleanup
sample_rules_file.unlink()
print(f"\n[OK] Validation complete, file cleaned up")

In [ ]:
# Import REP for integration
from mcp_server.protocols.rep.rep import AdvancedREP

rep = AdvancedREP()

# Define rules to validate and use
rules_data = {
    "rules": [
        [["All machines need maintenance", "Robot A is a machine"],
         "Robot A needs maintenance"],
        [["If maintenance overdue, schedule service", "Robot A maintenance overdue"],
         "Schedule service for Robot A"]
    ]
}

print("Step 1: Validate rules with VVP")
print("=" * 80)

# Validate first
vvp_result = vvp.execute(rules_data)
print(f"VVP Status: {vvp_result.get('status', 'Unknown')}")
print(f"Valid Rules: {vvp_result.get('valid_rules', 0)}/{vvp_result.get('total_rules', 0)}")

if vvp_result.get("status") == "valid":
    print("\n[OK] Rules validated successfully")

    print("\nStep 2: Use validated rules in REP deductive reasoning")
    print("=" * 80)

    # Extract validated rules for REP
    facts = [
        "All machines need maintenance",
        "Robot A is a machine",
        "If maintenance overdue, schedule service",
        "Robot A maintenance overdue"
    ]

    rules = [
        [["All machines need maintenance", "Robot A is a machine"],
         "Robot A needs maintenance"],
        [["If maintenance overdue, schedule service", "Robot A maintenance overdue"],
         "Schedule service for Robot A"]
    ]

    # Perform deductive reasoning with validated rules
    rep_result = rep.perform_reasoning(
        reasoning_type="deductive",
        facts=facts,
        rules=rules,
        context="Maintenance scheduling system"
    )

    print(f"REP Status: Success")
    print(f"Conclusions: {rep_result.get('conclusions', [])}")

    print("\n[OK] VVP + REP Integration Complete")
    print("  Rules were validated by VVP before being used in REP reasoning")
else:
    print("\n[X] Rules failed validation - cannot proceed with reasoning")
    print(f"  Errors: {vvp_result.get('errors', [])}")

## Summary

This tutorial demonstrated SIM-ONE's VVP protocol for rule validation:

### Key Capabilities

1. **Structure Validation** - Ensures rules follow proper format
2. **Conflict Detection** - Identifies contradicting rules
3. **Completeness Checking** - Verifies rule coverage
4. **Pre-Reasoning Validation** - Validates rules before REP usage

### When to Use VVP

- **Before deductive reasoning** - Validate rules before applying them
- **Rule base development** - Check rules during expert system creation
- **Policy verification** - Ensure business rules are consistent
- **Logic debugging** - Find errors in logical rule sets

### VVP Validation Checks

| Check Type | Purpose | Detects |
|------------|---------|---------|
| **Structure** | Syntax correctness | Empty premises, missing conclusions, wrong format |
| **Conflicts** | Logical consistency | Contradicting rules, circular dependencies |
| **Completeness** | Coverage | Missing cases, gaps in rule coverage |
| **Semantics** | Logical validity | Nonsensical rules, impossible conditions |

### API Reference

```python
from mcp_server.protocols.vvp.vvp import VVP

vvp = VVP()

# Validate rules
result = vvp.execute({
    "rules": [
        [[premises_list], conclusion],
        ...
    ],
    "context": {  # optional
        "domain": "...",
        "expected_coverage": [...]
    }
})
```

### Output Structure

```json
{
  "status": "valid" | "invalid" | "warning",
  "total_rules": 5,
  "valid_rules": 4,
  "invalid_rules": 1,
  "validation_details": [
    {
      "rule_index": 0,
      "status": "valid",
      "structure_valid": true,
      "semantics_valid": true,
      "message": "..."
    }
  ],
  "conflicts": [...],
  "errors": [...],
  "warnings": [...],
  "suggestions": [...]
}
```

### Integration Pattern

```
1. Define Rules
   (down)
2. VVP Validation <- You are here
   (down)
3. If Valid -> REP Deductive Reasoning
   (down)
4. Logical Conclusions
```

### Best Practices

1. **Always validate before reasoning** - Catch errors early
2. **Check for conflicts** - Prevent contradictory conclusions
3. **Verify completeness** - Ensure all cases are covered
4. **Provide context** - Help VVP understand domain-specific rules
5. **Review suggestions** - Improve rule quality based on VVP feedback

## Summary

This tutorial demonstrated SIM-ONE's VVP protocol for rule validation:

### Key Capabilities

1. **Structure Validation** - Ensures rules follow proper format
2. **Conflict Detection** - Identifies contradicting rules
3. **Completeness Checking** - Verifies rule coverage
4. **Pre-Reasoning Validation** - Validates rules before REP usage

### When to Use VVP

- **Before deductive reasoning** - Validate rules before applying them
- **Rule base development** - Check rules during expert system creation
- **Policy verification** - Ensure business rules are consistent
- **Logic debugging** - Find errors in logical rule sets

### VVP Validation Checks

| Check Type | Purpose | Detects |
|------------|---------|---------|
| **Structure** | Syntax correctness | Empty premises, missing conclusions, wrong format |
| **Conflicts** | Logical consistency | Contradicting rules, circular dependencies |
| **Completeness** | Coverage | Missing cases, gaps in rule coverage |
| **Semantics** | Logical validity | Nonsensical rules, impossible conditions |

### API Reference

```python
from mcp_server.protocols.vvp.vvp import VVP

vvp = VVP()

# Validate rules
result = vvp.execute({
    "rules": [
        [[premises_list], conclusion],
        ...
    ],
    "context": {  # optional
        "domain": "...",
        "expected_coverage": [...]
    }
})
```

### Output Structure

```json
{
  "status": "valid" | "invalid" | "warning",
  "total_rules": 5,
  "valid_rules": 4,
  "invalid_rules": 1,
  "validation_details": [
    {
      "rule_index": 0,
      "status": "valid",
      "structure_valid": true,
      "semantics_valid": true,
      "message": "..."
    }
  ],
  "conflicts": [...],
  "errors": [...],
  "warnings": [...],
  "suggestions": [...]
}
```

### Integration Pattern

```
1. Define Rules
   (down)
2. VVP Validation <- You are here
   (down)
3. If Valid -> REP Deductive Reasoning
   (down)
4. Logical Conclusions
```

### Best Practices

1. **Always validate before reasoning** - Catch errors early
2. **Check for conflicts** - Prevent contradictory conclusions
3. **Verify completeness** - Ensure all cases are covered
4. **Provide context** - Help VVP understand domain-specific rules
5. **Review suggestions** - Improve rule quality based on VVP feedback

## Debugging Common Rule Issues

This guide helps you quickly diagnose and fix common rule validation problems.

### Common Validation Errors

#### 1. "Empty premises list"

**Error**: `"Empty premises list - at least one premise required"`

**Cause**: Rule has `[]` for premises
```python
[[], "conclusion"]  # [X] Wrong
```

**Fix**: Add at least one premise
```python
[["at least one premise"], "conclusion"]  # [OK] Correct
```

**Why it matters**: Rules need premises to establish logical basis for conclusions.

---

#### 2. "Missing conclusion"

**Error**: `"Missing or empty conclusion"`

**Cause**: Conclusion is empty string or missing
```python
[["premise"], ""]        # [X] Empty string
[["premise"]]            # [X] Missing conclusion
```

**Fix**: Add non-empty conclusion
```python
[["premise"], "clear conclusion"]  # [OK] Correct
```

**Why it matters**: Rules must produce a result (the conclusion).

---

#### 3. "Wrong structure"

**Error**: `"Wrong structure - premises must be a list"`

**Cause**: Premises not in nested list
```python
["premise", "conclusion"]  # [X] Flat list
```

**Fix**: Wrap premises in nested list
```python
[["premise"], "conclusion"]  # [OK] Correct
```

**Why it matters**: VVP expects strict `[[[premises], conclusion]]` format.

---

#### 4. "Conflicting rules"

**Error**: `"Contradiction detected between rules X and Y"`

**Cause**: Same premises lead to different conclusions
```python
[["X"], "Y is true"]
[["X"], "Y is false"]  # [X] Conflict
```

**Fix Options**:
```python
# Option 1: Add distinguishing premises
[["X", "A"], "Y is true"]
[["X", "B"], "Y is false"]  # [OK] Now distinct

# Option 2: Remove incorrect rule
[["X"], "Y is true"]  # [OK] Keep only correct one
```

**Why it matters**: Conflicting rules produce contradictory conclusions, breaking logical consistency.

---

#### 5. "Incomplete coverage"

**Error**: `"Incomplete coverage: missing rules for <area>"`

**Cause**: Expected domain areas lack rules
```python
# context = {"expected_coverage": ["A", "B", "C"]}
rules = [
    [["A case"], "A result"],
    [["B case"], "B result"]
    # Missing "C" rules
]
```

**Fix**: Add rules for missing areas
```python
rules = [
    [["A case"], "A result"],
    [["B case"], "B result"],
    [["C case"], "C result"]  # [OK] Added
]
```

**Why it matters**: Incomplete rule sets may fail to handle all scenarios.

---

#### 6. "Invalid JSON format" (file-based validation)

**Error**: `"Failed to parse rules JSON file"`

**Cause**: JSON syntax errors in file
```json
{
    "rules": [
        [["premise"], "conclusion"],  // [X] Trailing comma
    ]
}
```

**Fix**: Use valid JSON
```json
{
    "rules": [
        [["premise"], "conclusion"]
    ]
}
```

**Why it matters**: VVP cannot validate rules it cannot parse.

### Systematic Debugging Workflow

```
+-------------------------------------+
| 1. Run VVP Validation               |
+--------------+-----------------------+
               (down)
+-------------------------------------+
| 2. Check status field               |
|    - "valid" -> Done [OK]           |
|    - "invalid" -> Continue to step 3|
|    - "warning" -> Continue to step 6|
+--------------+-----------------------+
               (down)
+-------------------------------------+
| 3. Fix Structural Errors            |
|    Read errors list                 |
|    For each error:                  |
|    - Find rule by index             |
|    - Identify error type            |
|    - Apply fix from guide above     |
+--------------+-----------------------+
               (down)
+-------------------------------------+
| 4. Re-run Validation                |
|    Check if structural errors gone  |
+--------------+-----------------------+
               (down)
+-------------------------------------+
| 5. Check for Warnings               |
|    If status == "valid", done [OK]  |
|    If status == "warning" -> step 6 |
+--------------+-----------------------+
               (down)
+-------------------------------------+
| 6. Address Warnings                 |
|    Review conflicts list            |
|    Review completeness              |
|    Decide if warnings acceptable    |
|    Apply fixes if needed            |
+--------------+-----------------------+
               (down)
+-------------------------------------+
| 7. Final Validation                 |
|    Confirm status acceptable        |
|    Ready for use [OK]               |
+-------------------------------------+
```

### Quick Reference: Error -> Fix

| Error | Quick Fix |
|-------|-----------|
| Empty premises | Add `["at least one premise"]` |
| Missing conclusion | Add `"conclusion string"` |
| Wrong structure | Wrap premises: `[["..."], "..."]` |
| Conflicting rules | Add distinguishing premises or remove rule |
| Incomplete coverage | Add rules for missing domain areas |
| Invalid JSON | Check JSON syntax, remove trailing commas |

### Testing After Fixes

After fixing errors, verify the fix worked:

```python
# Run validation
result = vvp.execute(rules=fixed_rules)

# Check result
assert result["status"] in ["valid", "warning"], f"Still invalid: {result['errors']}"
assert result["invalid_rules"] == 0, "Still have invalid rules"

# If using in REP, test integration
if result["status"] == "valid":
    rep_result = rep.perform_reasoning(
        reasoning_type="deductive",
        facts=test_facts,
        rules=fixed_rules
    )
    # Verify reasoning works as expected
```

### When to Ask for Help

If you encounter:
- Persistent errors after applying fixes
- Conflicts you don't understand how to resolve
- Completeness warnings when you believe coverage is adequate
- Rules that should be valid but VVP marks as invalid

**Next steps**:
1. Check VVP documentation for edge cases
2. Verify rule format matches examples exactly
3. Test with minimal example to isolate issue
4. Review validation_details for per-rule feedback

## Advanced VVP Usage

For power users and production deployments, VVP offers advanced capabilities for handling complex rule sets and specialized validation needs.

### 1. Performance Optimization for Large Rule Sets

#### Challenge
Rule sets with 100+ rules can have slower validation times due to:
- Conflict detection (O(n^2) comparisons)
- Completeness checking across all rules
- Detailed per-rule validation

#### Solutions

**Batch Validation**:
```python
# Instead of validating entire rule set repeatedly
# Split into logical groups and validate incrementally

# Validate core rules once
core_validation = vvp.execute(rules=core_rules)

# Add and validate new rules against validated core
new_rules_validation = vvp.execute(
    rules=core_rules + new_rules
)
```

**Rule Set Partitioning**:
```python
# Partition rules by domain
medical_rules = [...]  # rules related to medical diagnosis
treatment_rules = [...]  # rules related to treatment

# Validate separately (parallel if needed)
medical_result = vvp.execute(rules=medical_rules, context={"domain": "diagnosis"})
treatment_result = vvp.execute(rules=treatment_rules, context={"domain": "treatment"})
```

**Caching Validation Results**:
```python
import hashlib
import json

def get_rules_hash(rules):
    """Generate hash of rule set for caching"""
    return hashlib.sha256(json.dumps(rules, sort_keys=True).encode()).hexdigest()

# Check cache before validating
rules_hash = get_rules_hash(rules)
if rules_hash in validation_cache:
    result = validation_cache[rules_hash]
else:
    result = vvp.execute(rules=rules)
    validation_cache[rules_hash] = result
```

### 2. Domain-Specific Validation

#### Custom Context for Specialized Domains

VVP's context parameter can be extended for domain-specific needs:

```python
# Medical domain validation
medical_context = {
    "domain": "medical_diagnosis",
    "expected_coverage": [
        "symptom_identification",
        "differential_diagnosis",
        "primary_diagnosis",
        "treatment_recommendation"
    ]
}

result = vvp.execute(rules=medical_rules, context=medical_context)
```

#### Domain-Specific Validation Layers

Add custom validation on top of VVP:

```python
def validate_medical_rules(rules):
    """Add medical-specific validation"""
    # First, standard VVP validation
    vvp_result = vvp.execute(rules=rules)
    
    if vvp_result["status"] == "invalid":
        return vvp_result
    
    # Then, medical-specific checks
    medical_checks = {
        "requires_patient_consent": [],
        "requires_medical_license": [],
        "drug_interaction_safe": []
    }
    
    for i, rule in enumerate(rules):
        premises, conclusion = rule
        # Check medical-specific requirements
        if "prescribe" in conclusion.lower():
            medical_checks["requires_medical_license"].append(i)
        # ... more checks
    
    # Merge with VVP result
    vvp_result["medical_validation"] = medical_checks
    return vvp_result
```

### 3. Incremental Rule Development

#### Development Workflow

```python
# Stage 1: Start with minimal rules
rules_v1 = [
    [["symptom: fever"], "possible: infection"]
]

validation_v1 = vvp.execute(
    rules=rules_v1,
    context={"expected_coverage": ["symptoms", "diagnosis", "treatment"]}
)
# Shows: 33% coverage (only symptoms)

# Stage 2: Add diagnosis rules
rules_v2 = rules_v1 + [
    [["possible: infection", "test: positive"], "diagnosis: bacterial_infection"]
]

validation_v2 = vvp.execute(rules=rules_v2, context=...)
# Shows: 66% coverage (symptoms + diagnosis)

# Stage 3: Add treatment rules
rules_v3 = rules_v2 + [
    [["diagnosis: bacterial_infection"], "treatment: antibiotics"]
]

validation_v3 = vvp.execute(rules=rules_v3, context=...)
# Shows: 100% coverage
```

#### Version Tracking

```python
rule_versions = {
    "v1.0": {"rules": rules_v1, "validation": validation_v1},
    "v1.1": {"rules": rules_v2, "validation": validation_v2},
    "v2.0": {"rules": rules_v3, "validation": validation_v3}
}

# Compare validation results across versions
for version, data in rule_versions.items():
    coverage = data['validation'].get('completeness', {}).get('coverage_percentage', 'N/A')
    print(f"{version}: {data['validation']['status']}, Coverage: {coverage}%")
```

### 4. Rule Refactoring Based on VVP Feedback

#### Using VVP Results to Improve Rules

**Conflict Resolution**:
```python
result = vvp.execute(rules=rules)

if result["conflicts"]:
    for conflict in result["conflicts"]:
        rule_indices = conflict["rule_indices"]
        # Identify conflicting rules
        rule1 = rules[rule_indices[0]]
        rule2 = rules[rule_indices[1]]
        
        # Strategy: Add distinguishing premises
        # Modify rules to be more specific
        # Re-validate after changes
```

**Coverage-Driven Development**:
```python
result = vvp.execute(rules=rules, context=context)

if result.get("completeness", {}).get("coverage_percentage", 100) < 100:
    missing = result["completeness"]["missing"]
    print(f"Need to add rules for: {missing}")
    
    # For each missing area, add rules
    for area in missing:
        new_rule = create_rule_for_area(area)
        rules.append(new_rule)
    
    # Re-validate
    result = vvp.execute(rules=rules, context=context)
```

### 5. CI/CD Integration

#### Automated Rule Validation Pipeline

```python
# validate_rules.py
import sys
import json
from mcp_server.protocols.vvp.vvp import VVP

def ci_validation(rules_file):
    """Validation for CI/CD pipeline"""
    with open(rules_file) as f:
        rules_data = json.load(f)
    
    vvp = VVP()
    result = vvp.execute(rules=rules_data["rules"])
    
    # Fail CI if rules invalid
    if result["status"] == "invalid":
        print(f"[X] Validation failed: {result['errors']}")
        sys.exit(1)
    
    # Warn on conflicts but don't fail
    if result["status"] == "warning":
        print(f"[!] Warnings: {result['warnings']}")
    
    print("[OK] Rules validated successfully")
    return result

if __name__ == "__main__":
    ci_validation(sys.argv[1])
```

### 6. Rule Quality Metrics

#### Defining Quality Thresholds

```python
def assess_rule_quality(validation_result):
    """Assess overall rule set quality"""
    score = 0
    feedback = []
    
    # Structural validity (40 points)
    if validation_result["status"] in ["valid", "warning"]:
        score += 40
        feedback.append("[OK] All rules structurally valid")
    
    # No conflicts (30 points)
    if not validation_result.get("conflicts"):
        score += 30
        feedback.append("[OK] No rule conflicts")
    else:
        feedback.append(f"[!] {len(validation_result['conflicts'])} conflicts found")
    
    # Completeness (30 points)
    coverage = validation_result.get("completeness", {}).get("coverage_percentage", 0)
    score += int(coverage * 0.3)
    feedback.append(f"Coverage: {coverage}%")
    
    # Overall assessment
    if score >= 90:
        grade = "A - Production Ready"
    elif score >= 75:
        grade = "B - Good"
    elif score >= 60:
        grade = "C - Needs Improvement"
    else:
        grade = "D - Significant Issues"
    
    return {"score": score, "grade": grade, "feedback": feedback}

# Usage
result = vvp.execute(rules=rules, context=context)
quality = assess_rule_quality(result)
print(f"Rule Quality: {quality['grade']} ({quality['score']}/100)")
for item in quality['feedback']:
    print(f"  {item}")
```

### 7. Migration from Other Rule Formats

#### Converting Prolog-style Rules

```python
def prolog_to_vvp(prolog_rule):
    """Convert Prolog rule to VVP format"""
    # Example: "mortal(X) :- human(X)."
    # Becomes: [["human(X)"], "mortal(X)"]
    
    if ":-" in prolog_rule:
        conclusion, premises_str = prolog_rule.split(":-")
        premises = [p.strip() for p in premises_str.replace(".", "").split(",")]
        return [premises, conclusion.strip()]
    else:
        # Fact (no premises)
        return [["fact"], prolog_rule.strip().replace(".", "")]

# Validate converted rules
prolog_rules = ["mortal(X) :- human(X).", "human(socrates)."]
vvp_rules = [prolog_to_vvp(r) for r in prolog_rules]
result = vvp.execute(rules=vvp_rules)
```

### When to Use Advanced Features

| Feature | Use When |
|---------|----------|
| **Performance optimization** | Rule sets > 100 rules, validation in hot path |
| **Domain-specific validation** | Industry-specific requirements (medical, legal, financial) |
| **Incremental development** | Building large rule sets iteratively |
| **VVP-driven refactoring** | Improving existing rule sets systematically |
| **CI/CD integration** | Production deployments, team collaboration |
| **Quality metrics** | Need objective measures of rule set quality |
| **Format migration** | Converting from other rule engines to VVP |

In [ ]:
# Define rules with various structural issues
invalid_rules = {
    "rules": [
        # Valid rule
        [["All cats are animals"], "Fluffy is an animal"],

        # INVALID: Empty premises list
        [[], "Something is true"],

        # INVALID: Missing conclusion (empty string)
        [["Some premise"], ""],

        # INVALID: Wrong structure (not nested list)
        ["premise", "conclusion"],

        # Valid rule
        [["X implies Y", "X is true"], "Y is true"]
    ]
}

# Validate rules
result = vvp.execute(invalid_rules)

print("VVP Validation Result (Invalid Rules):")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")
print(f"Valid Rules: {result.get('valid_rules', 0)}")
print(f"Invalid Rules: {result.get('invalid_rules', 0)}")

if result.get("validation_details"):
    print(f"\n📋 Detailed Validation:")
    for i, detail in enumerate(result["validation_details"], 1):
        status_symbol = "✓" if detail.get("status") == "valid" else "✗"
        print(f"  {status_symbol} Rule {i}: {detail.get('status', 'Unknown')}")
        if detail.get("error"):
            print(f"    Error: {detail['error']}")
        if detail.get("message"):
            print(f"    Message: {detail['message']}")

if result.get("errors"):
    print(f"\n❌ Errors Summary:")
    for error in result["errors"]:
        print(f"  - {error}")

## Example 3: Detecting Rule Conflicts

VVP can identify when rules contradict each other.

In [ ]:
# Define rules that conflict with each other
conflicting_rules = {
    "rules": [
        [["All birds can fly"], "Penguins can fly"],
        [["Penguins are birds"], "Penguins exist"],
        [["Penguins cannot fly"], "Penguins are flightless"],  # Conflicts with rule 1
        [["If X can fly, X has wings"], "Penguins have wings"]
    ]
}

# Validate rules
result = vvp.execute(conflicting_rules)

print("VVP Conflict Detection:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")

if result.get("conflicts"):
    print(f"\n⚠ Conflicts Detected ({len(result['conflicts'])}):")
    for i, conflict in enumerate(result["conflicts"], 1):
        print(f"\n  Conflict {i}:")
        print(f"    Rule A: {conflict.get('rule_a')}")
        print(f"    Rule B: {conflict.get('rule_b')}")
        print(f"    Issue: {conflict.get('description')}")
else:
    print("\n✓ No conflicts detected")

if result.get("warnings"):
    print(f"\n💡 Warnings:")
    for warning in result["warnings"]:
        print(f"  - {warning}")

## Example 4: Checking Rule Set Completeness

Ensure rule sets cover all necessary cases without gaps.

In [ ]:
# Define rule set for checking completeness
rules_with_gaps = {
    "rules": [
        [["Temperature > 0°C", "Pressure = 1 atm"], "Water is liquid"],
        [["Temperature > 100°C", "Pressure = 1 atm"], "Water is gas"],
        # Missing: What about temperature < 0°C? (water is solid/ice)
    ],
    "context": {
        "domain": "Water phase transitions",
        "expected_coverage": ["solid", "liquid", "gas"]
    }
}

# Validate rules
result = vvp.execute(rules_with_gaps)

print("VVP Completeness Check:")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")

if result.get("completeness"):
    comp = result["completeness"]
    print(f"\nCompleteness Analysis:")
    print(f"  Coverage: {comp.get('coverage_percentage', 0):.1f}%")
    print(f"  Covered Cases: {comp.get('covered_cases', [])}")
    print(f"  Missing Cases: {comp.get('missing_cases', [])}")

if result.get("suggestions"):
    print(f"\n💡 Suggestions for Improvement:")
    for suggestion in result["suggestions"]:
        print(f"  - {suggestion}")

## Example 5: Validating Rules from Files

VVP can validate rules stored in JSON files.

In [ ]:
# Create sample rules file
sample_rules_file = Path("sample_rules.json")

sample_rules = {
    "rules": [
        [["All employees must clock in", "John is an employee"], "John must clock in"],
        [["Meeting starts at 9 AM", "It is 9 AM"], "Meeting is starting"],
        [["If urgent, escalate immediately"], "Handle urgent matters promptly"]
    ]
}

# Write to file
sample_rules_file.write_text(json.dumps(sample_rules, indent=2))

# Validate rules from file
rules_from_file = json.loads(sample_rules_file.read_text())
result = vvp.execute(rules_from_file)

print(f"VVP Validation from File: {sample_rules_file}")
print("=" * 80)

print(f"Status: {result.get('status', 'Unknown')}")
print(f"Total Rules: {result.get('total_rules', 0)}")
print(f"Valid Rules: {result.get('valid_rules', 0)}")

# Create validation summary table
if result.get("validation_details"):
    summary_data = []
    for i, detail in enumerate(result["validation_details"], 1):
        rule_text = f"Rule {i}"
        if detail.get("rule"):
            premises = detail["rule"].get("premises", [])
            conclusion = detail["rule"].get("conclusion", "")
            rule_text = f"{' & '.join(premises[:2])} → {conclusion[:30]}"

        summary_data.append({
            "Rule": f"Rule {i}",
            "Status": detail.get("status", "Unknown"),
            "Structure": "✓" if detail.get("structure_valid") else "✗",
            "Semantics": "✓" if detail.get("semantics_valid") else "✗"
        })

    df = pd.DataFrame(summary_data)
    print("\n" + df.to_string(index=False))

# Cleanup
sample_rules_file.unlink()
print(f"\n✓ Validation complete, file cleaned up")

## Example 6: VVP + REP Integration

Demonstrate how VVP validates rules before they're used in REP deductive reasoning.

In [ ]:
# Import REP for integration
from mcp_server.protocols.rep.rep import AdvancedREP

rep = AdvancedREP()

# Define rules to validate and use
rules_data = {
    "rules": [
        [["All machines need maintenance", "Robot A is a machine"],
         "Robot A needs maintenance"],
        [["If maintenance overdue, schedule service", "Robot A maintenance overdue"],
         "Schedule service for Robot A"]
    ]
}

print("Step 1: Validate rules with VVP")
print("=" * 80)

# Validate first
vvp_result = vvp.execute(rules_data)
print(f"VVP Status: {vvp_result.get('status', 'Unknown')}")
print(f"Valid Rules: {vvp_result.get('valid_rules', 0)}/{vvp_result.get('total_rules', 0)}")

if vvp_result.get("status") == "valid":
    print("\n✓ Rules validated successfully")

    print("\nStep 2: Use validated rules in REP deductive reasoning")
    print("=" * 80)

    # Extract validated rules for REP
    facts = [
        "All machines need maintenance",
        "Robot A is a machine",
        "If maintenance overdue, schedule service",
        "Robot A maintenance overdue"
    ]

    rules = [
        [["All machines need maintenance", "Robot A is a machine"],
         "Robot A needs maintenance"],
        [["If maintenance overdue, schedule service", "Robot A maintenance overdue"],
         "Schedule service for Robot A"]
    ]

    # Perform deductive reasoning with validated rules
    rep_result = rep.perform_reasoning(
        reasoning_type="deductive",
        facts=facts,
        rules=rules,
        context="Maintenance scheduling system"
    )

    print(f"REP Status: Success")
    print(f"Conclusions: {rep_result.get('conclusions', [])}")

    print("\n✓ VVP + REP Integration Complete")
    print("  Rules were validated by VVP before being used in REP reasoning")
else:
    print("\n✗ Rules failed validation - cannot proceed with reasoning")
    print(f"  Errors: {vvp_result.get('errors', [])}")

## Summary

This tutorial demonstrated SIM-ONE's VVP protocol for rule validation:

### Key Capabilities

1. **Structure Validation** - Ensures rules follow proper format
2. **Conflict Detection** - Identifies contradicting rules
3. **Completeness Checking** - Verifies rule coverage
4. **Pre-Reasoning Validation** - Validates rules before REP usage

### When to Use VVP

- **Before deductive reasoning** - Validate rules before applying them
- **Rule base development** - Check rules during expert system creation
- **Policy verification** - Ensure business rules are consistent
- **Logic debugging** - Find errors in logical rule sets

### VVP Validation Checks

| Check Type | Purpose | Detects |
|------------|---------|---------||
| **Structure** | Syntax correctness | Empty premises, missing conclusions, wrong format |
| **Conflicts** | Logical consistency | Contradicting rules, circular dependencies |
| **Completeness** | Coverage | Missing cases, gaps in rule coverage |
| **Semantics** | Logical validity | Nonsensical rules, impossible conditions |

### API Reference

```python
from mcp_server.protocols.vvp.vvp import VVP

vvp = VVP()

# Validate rules
result = vvp.execute({
    "rules": [
        [[premises_list], conclusion],
        ...
    ],
    "context": {  # optional
        "domain": "...",
        "expected_coverage": [...]
    }
})
```

### Output Structure

```json
{
  "status": "valid" | "invalid" | "warning",
  "total_rules": 5,
  "valid_rules": 4,
  "invalid_rules": 1,
  "validation_details": [
    {
      "rule_index": 0,
      "status": "valid",
      "structure_valid": true,
      "semantics_valid": true,
      "message": "..."
    }
  ],
  "conflicts": [...],
  "errors": [...],
  "warnings": [...],
  "suggestions": [...]
}
```

### Integration Pattern

```
1. Define Rules
   ↓
2. VVP Validation ← You are here
   ↓
3. If Valid → REP Deductive Reasoning
   ↓
4. Logical Conclusions
```

### Best Practices

1. **Always validate before reasoning** - Catch errors early
2. **Check for conflicts** - Prevent contradictory conclusions
3. **Verify completeness** - Ensure all cases are covered
4. **Provide context** - Help VVP understand domain-specific rules
5. **Review suggestions** - Improve rule quality based on VVP feedback